In [1]:
%load_ext autoreload
%autoreload 2

# Korpuse verbide statistika kogumine


Kogutakse kokku statistika:
* verbide esinemised <code>verbs.tsv</code>;
* verbide esinemised eristatuna lihtverbiks ja liitverbiks <code>verbs_afiksaaladverb.tsv</code> ;
* verbide koguarv.

In [3]:
from notebook_context import corpus_reader, LISTS_FOLDER
from src.syntax.syntax_graph import SyntaxGraph
from src.syntax.utils import ListUtils
import pandas as pd

VERBS_LIST = LISTS_FOLDER / "stats/verbs.tsv"
VERBS_AFIKS_LIST = LISTS_FOLDER / "stats/verbs_afiksaaladverb.tsv"

In [8]:
%%time
stats = {}
# hoitakse algusest lõpuni mälus
verb_plain_stat = {}
verbs_compound_stat = {}
verbs_total = 0


count = 0
for collection_id, graph in corpus_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # eraldame ainult verbid:
    verb_nodes = graph.get_nodes_by_attributes(attrname="POS", attrvalue="VERB")
    # compound:prt
    compound_nodes = graph.get_nodes_by_attributes(
        attrname="deprel", attrvalue="compound:prt"
    )
    for verb in verb_nodes:
        verbs_total += 1
        lemma = graph.nodes[verb]['lemma']
        if not lemma in verb_plain_stat:
            verb_plain_stat[lemma] = 0
        verb_plain_stat[lemma] += 1

        # compound 
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        # compound children
        n_compounds = ListUtils.list_intersection(kids, compound_nodes)
        if not len(n_compounds):
            verb_compound = ""
            n_compounds.append(None)
        else:
            verb_compound = ", ".join(
                [graph.nodes[n]["lemma"] for n in sorted(n_compounds) if n]
            )
        key = (lemma, verb_compound, )
        if not key in verbs_compound_stat:
            verbs_compound_stat[key] = 0
        verbs_compound_stat[key] += 1
        
print('Verbs total:', verbs_total)
 

../data/vrt-with-meta-corpus-02-06-25_ordered.vrt
Verbs total: 218751
CPU times: user 34 s, sys: 455 ms, total: 34.4 s
Wall time: 34.4 s


In [9]:
verb_plain_list = [(key, value) for key, value in verb_plain_stat.items()]
df_plain = pd.DataFrame(verb_plain_list, columns=["verb", "total"])
df_plain.head()

df_plain.to_csv(VERBS_LIST, index=None, sep="\t")

In [10]:
verbs_compound_list = [
    (key[0], key[1], value) for key, value in verbs_compound_stat.items()
]
df_compound = pd.DataFrame(verbs_compound_list, columns=["verb", "compound", "total"])
display(df_compound.head())

df_compound.to_csv(VERBS_AFIKS_LIST, index=None, sep="\t")

,verb,compound,total
0,õtlema,,81
1,rääkima,ära,40
2,vaatama,,2028
3,erinema,,199
4,õppima,,4668
